# Methodology

## Objective
Predict a continuous spoken-English grammar score from 0–5 using audio recordings.

## Pipeline
The solution combines three complementary information sources:

1. **Acoustic / Prosodic Features**
   - Duration
   - RMS energy
   - Zero-crossing rate
   - Spectral centroid, bandwidth and rolloff
   - MFCC statistics
   - Pitch statistics
   - Voicing ratio
   - Low-energy ratio
   - CatBoost regression

2. **Whisper Speech Representation**
   - Full-clip Whisper transcription
   - Whisper encoder embeddings
   - Mean pooling of encoder representations
   - Ridge regression

3. **Transcript Features**
   - Word-level TF-IDF using unigrams and bigrams
   - Ridge regression

## Evaluation
A fixed 5-fold K-Fold cross-validation strategy was used.
Model predictions were generated using out-of-fold (OOF) predictions.

Metrics:
- Pearson correlation: higher is better
- RMSE: lower is better

## Final Ensemble
The selected ensemble combines:
- 50% Acoustic / CatBoost
- 45% Whisper encoder / Ridge
- 5% Word TF-IDF / Ridge

The final predictions are clipped to the valid 0–5 score range and written to `submission.csv`.

In [1]:
# ============================================================
# SHL Hiring Assessment 2026 — Setup
# ============================================================

import os
import re
import gc
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Dataset paths
BASE_DIR = Path(
    "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"
)

TRAIN_CSV = BASE_DIR / "train.csv"
TEST_CSV = BASE_DIR / "test.csv"

TRAIN_AUDIO_DIR = BASE_DIR / "train"
TEST_AUDIO_DIR = BASE_DIR / "test"

# Working/cache directory
WORK_DIR = Path("/kaggle/working")
CACHE_DIR = WORK_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("SHL GRAMMAR SCORING ENGINE")
print("=" * 70)
print("Base directory :", BASE_DIR)
print("Train CSV      :", TRAIN_CSV)
print("Test CSV       :", TEST_CSV)
print("Cache directory:", CACHE_DIR)

SHL GRAMMAR SCORING ENGINE
Base directory : /kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final
Train CSV      : /kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/train.csv
Test CSV       : /kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test.csv
Cache directory: /kaggle/working/cache


In [2]:
# ============================================================
# CELL 2 — LOAD AND VALIDATE DATA
# ============================================================

train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)

print("=" * 70)
print("DATASET")
print("=" * 70)

print("\nTrain shape:", train.shape)
print("Test shape :", test.shape)

print("\nTrain columns:", train.columns.tolist())
print("Test columns :", test.columns.tolist())

print("\nMissing values:")
print("Train:\n", train.isna().sum())
print("Test:\n", test.isna().sum())

print("\nDuplicate filenames:")
print("Train:", train["filename"].duplicated().sum())
print("Test :", test["filename"].duplicated().sum())

print("\nLabel statistics:")
print(train["label"].describe())

print("\nLabel distribution:")
print(train["label"].value_counts().sort_index())

# Verify audio counts
train_audio_count = len(list(TRAIN_AUDIO_DIR.glob("*.wav")))
test_audio_count = len(list(TEST_AUDIO_DIR.glob("*.wav")))

print("\nAudio files:")
print("Train audio:", train_audio_count)
print("Test audio :", test_audio_count)

# Critical consistency checks
assert train.shape[0] == 769
assert test.shape[0] == 216
assert train["filename"].is_unique
assert test["filename"].is_unique
assert train["label"].notna().all()
assert train_audio_count == len(train)
assert test_audio_count == len(test)

print("\n" + "=" * 70)
print("✅ DATA VALIDATION PASSED")
print("=" * 70)

DATASET

Train shape: (769, 2)
Test shape : (216, 2)

Train columns: ['filename', 'label']
Test columns : ['filename', 'label']

Missing values:
Train:
 filename    0
label       0
dtype: int64
Test:
 filename    0
label       0
dtype: int64

Duplicate filenames:
Train: 0
Test : 0

Label statistics:
count    769.000000
mean       3.313394
std        1.239000
min        0.000000
25%        2.500000
50%        3.000000
75%        4.000000
max        5.000000
Name: label, dtype: float64

Label distribution:
label
0.0     37
1.0      1
1.5      3
2.0    102
2.5     79
3.0    174
3.5     64
4.0    124
4.5     52
5.0    133
Name: count, dtype: int64

Audio files:
Train audio: 769
Test audio : 216

✅ DATA VALIDATION PASSED


In [3]:
# ============================================================
# CELL 6 — FINAL CV SPLITS FOR REGRESSION
# ============================================================

from sklearn.model_selection import KFold

N_FOLDS = 5

kf = KFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)

train["fold"] = -1

for fold, (_, val_idx) in enumerate(kf.split(train)):
    train.loc[val_idx, "fold"] = fold

print("=" * 70)
print("FINAL 5-FOLD REGRESSION CV")
print("=" * 70)

print("\nFold sizes:")
print(train["fold"].value_counts().sort_index())

print("\nFold label means:")
print(train.groupby("fold")["label"].mean())

assert train["fold"].nunique() == 5
assert train["fold"].notna().all()

print("\n" + "=" * 70)
print("✅ FINAL CV SPLITS LOCKED")
print("=" * 70)

FINAL 5-FOLD REGRESSION CV

Fold sizes:
fold
0    154
1    154
2    154
3    154
4    153
Name: count, dtype: int64

Fold label means:
fold
0    3.217532
1    3.152597
2    3.461039
3    3.305195
4    3.431373
Name: label, dtype: float64

✅ FINAL CV SPLITS LOCKED


In [4]:
# ============================================================
# CELL 4 — EVALUATION UTILITIES
# ============================================================

from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr


def evaluate_predictions(y_true, y_pred):
    """
    Evaluate grammar-score predictions.

    Pearson: higher is better
    RMSE: lower is better
    """
    pearson = pearsonr(y_true, y_pred)[0]
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    return pearson, rmse


def print_scores(name, y_true, y_pred):
    pearson, rmse = evaluate_predictions(y_true, y_pred)

    print(f"{name}")
    print(f"Pearson : {pearson:.4f}")
    print(f"RMSE    : {rmse:.4f}")

    return pearson, rmse


y = train["label"].values

print("=" * 70)
print("EVALUATION UTILITIES READY")
print("=" * 70)

print("Target shape:", y.shape)
print("Target range:", y.min(), "→", y.max())

EVALUATION UTILITIES READY
Target shape: (769,)
Target range: 0.0 → 5.0


In [5]:
# ============================================================
# CELL 5 — CHECK TRANSCRIPT CACHE
# ============================================================

train_transcript_cache = CACHE_DIR / "train_transcripts.csv"
test_transcript_cache = CACHE_DIR / "test_transcripts.csv"

print("=" * 70)
print("TRANSCRIPT CACHE CHECK")
print("=" * 70)

print("Train cache:", train_transcript_cache)
print("Exists     :", train_transcript_cache.exists())

print("\nTest cache :", test_transcript_cache)
print("Exists     :", test_transcript_cache.exists())

if train_transcript_cache.exists() and test_transcript_cache.exists():
    train_transcripts = pd.read_csv(train_transcript_cache)
    test_transcripts = pd.read_csv(test_transcript_cache)

    print("\nCached train transcripts:", train_transcripts.shape)
    print("Cached test transcripts :", test_transcripts.shape)

    print("\nTrain columns:", train_transcripts.columns.tolist())
    print("Test columns :", test_transcripts.columns.tolist())

print("\n" + "=" * 70)

TRANSCRIPT CACHE CHECK
Train cache: /kaggle/working/cache/train_transcripts.csv
Exists     : False

Test cache : /kaggle/working/cache/test_transcripts.csv
Exists     : False



In [6]:
# ============================================================
# CELL 7 — WHISPER SETUP
# ============================================================

import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WHISPER_MODEL_NAME = "openai/whisper-small"

print("=" * 70)
print("WHISPER SETUP")
print("=" * 70)

print("Device:", DEVICE)
print("Model :", WHISPER_MODEL_NAME)

processor = WhisperProcessor.from_pretrained(
    WHISPER_MODEL_NAME
)

whisper_model = WhisperForConditionalGeneration.from_pretrained(
    WHISPER_MODEL_NAME
).to(DEVICE)

whisper_model.eval()

print("\nModel loaded successfully.")
print("GPU:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU")

print("\n" + "=" * 70)
print("✅ WHISPER READY")
print("=" * 70)

WHISPER SETUP
Device: cuda
Model : openai/whisper-small


preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]


Model loaded successfully.
GPU: Tesla T4

✅ WHISPER READY


In [7]:
# ============================================================
# CELL 8 — WHISPER SINGLE-AUDIO TEST
# ============================================================

import librosa
import time

test_file = TRAIN_AUDIO_DIR / train.iloc[0]["filename"]

print("=" * 70)
print("WHISPER SANITY TEST")
print("=" * 70)

print("File:", test_file.name)

audio, sr = librosa.load(
    test_file,
    sr=16000,
    mono=True
)

print("Duration:", round(len(audio) / sr, 2), "seconds")

start_time = time.time()

inputs = processor(
    audio,
    sampling_rate=16000,
    return_tensors="pt"
)

input_features = inputs.input_features.to(DEVICE)

with torch.no_grad():
    predicted_ids = whisper_model.generate(
        input_features
    )

transcript = processor.batch_decode(
    predicted_ids,
    skip_special_tokens=True
)[0].strip()

elapsed = time.time() - start_time

print("\nTranscript:")
print(transcript)

print("\nInference time:", round(elapsed, 2), "seconds")

print("\n" + "=" * 70)
print("✅ WHISPER SANITY TEST PASSED")
print("=" * 70)

WHISPER SANITY TEST
File: audio_192.wav


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.


Duration: 60.51 seconds


[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



Transcript:
The plate round is very, very beautiful and fun. There are places like the merry-go-round, the swing. The students usually enjoy the swing. It has to be like, it's more like a game of play.

Inference time: 2.87 seconds

✅ WHISPER SANITY TEST PASSED


In [8]:
# ============================================================
# CELL 9 — FULL WHOLE-CLIP WHISPER TRANSCRIPTION
# ============================================================

import time
import librosa
from tqdm.auto import tqdm

train_transcript_cache = CACHE_DIR / "train_transcripts.csv"
test_transcript_cache = CACHE_DIR / "test_transcripts.csv"


def transcribe_file(filepath):
    """Transcribe the complete audio clip."""

    audio, sr = librosa.load(
        filepath,
        sr=16000,
        mono=True
    )

    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    )

    input_features = inputs.input_features.to(DEVICE)

    with torch.no_grad():
        predicted_ids = whisper_model.generate(
            input_features,
            task="transcribe"
        )

    text = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0].strip()

    return text


def build_transcript_cache(df, audio_dir, cache_path, split_name):
    """
    Transcribe all files and save progress continuously.
    Existing completed files are reused.
    """

    if cache_path.exists():
        cache_df = pd.read_csv(cache_path)
        cache = dict(
            zip(cache_df["filename"], cache_df["transcript"])
        )
    else:
        cache = {}

    missing = [
        f for f in df["filename"]
        if f not in cache
    ]

    print("=" * 70)
    print(f"{split_name.upper()} TRANSCRIPTION")
    print("=" * 70)
    print("Total files :", len(df))
    print("Already done:", len(df) - len(missing))
    print("Remaining   :", len(missing))

    start_all = time.time()

    for i, filename in enumerate(
        tqdm(missing, desc=f"{split_name} Whisper")
    ):

        filepath = audio_dir / filename

        try:
            text = transcribe_file(filepath)
        except Exception as e:
            print(f"\nERROR: {filename} -> {e}")
            text = ""

        cache[filename] = text

        # Save progress after every file
        cache_df = pd.DataFrame({
            "filename": list(cache.keys()),
            "transcript": list(cache.values())
        })

        cache_df.to_csv(
            cache_path,
            index=False
        )

        if (i + 1) % 50 == 0:
            elapsed = time.time() - start_all
            print(
                f"\nProcessed this run: {i + 1}/{len(missing)}"
                f" | Elapsed: {elapsed/60:.1f} min"
            )

    result = pd.DataFrame({
        "filename": df["filename"],
        "transcript": [
            cache.get(f, "") for f in df["filename"]
        ]
    })

    print("\nCompleted:", split_name)
    print("Shape:", result.shape)
    print(
        "Empty transcripts:",
        (result["transcript"].str.strip() == "").sum()
    )

    return result


# Train
train_transcripts = build_transcript_cache(
    train,
    TRAIN_AUDIO_DIR,
    train_transcript_cache,
    "train"
)

# Test
test_transcripts = build_transcript_cache(
    test,
    TEST_AUDIO_DIR,
    test_transcript_cache,
    "test"
)

print("\n" + "=" * 70)
print("✅ ALL WHISPER TRANSCRIPTS READY")
print("=" * 70)

TRAIN TRANSCRIPTION
Total files : 769
Already done: 0
Remaining   : 769


train Whisper:   0%|          | 0/769 [00:00<?, ?it/s]


Processed this run: 50/769 | Elapsed: 1.1 min

Processed this run: 100/769 | Elapsed: 2.1 min

Processed this run: 150/769 | Elapsed: 3.1 min

Processed this run: 200/769 | Elapsed: 4.1 min

Processed this run: 250/769 | Elapsed: 5.4 min

Processed this run: 300/769 | Elapsed: 6.6 min

Processed this run: 350/769 | Elapsed: 7.8 min

Processed this run: 400/769 | Elapsed: 9.0 min

Processed this run: 450/769 | Elapsed: 9.9 min

Processed this run: 500/769 | Elapsed: 10.9 min

Processed this run: 550/769 | Elapsed: 11.9 min

Processed this run: 600/769 | Elapsed: 12.9 min

Processed this run: 650/769 | Elapsed: 13.9 min

Processed this run: 700/769 | Elapsed: 14.9 min

Processed this run: 750/769 | Elapsed: 16.0 min

Completed: train
Shape: (769, 2)
Empty transcripts: 0
TEST TRANSCRIPTION
Total files : 216
Already done: 0
Remaining   : 216


test Whisper:   0%|          | 0/216 [00:00<?, ?it/s]


Processed this run: 50/216 | Elapsed: 1.0 min

Processed this run: 100/216 | Elapsed: 2.1 min

Processed this run: 150/216 | Elapsed: 3.2 min

Processed this run: 200/216 | Elapsed: 4.3 min

Completed: test
Shape: (216, 2)
Empty transcripts: 0

✅ ALL WHISPER TRANSCRIPTS READY


In [9]:
# ============================================================
# CELL 10 — TRANSCRIPT QUALITY AUDIT
# ============================================================

# Merge transcripts with labels
train_text = train.merge(
    train_transcripts,
    on="filename",
    how="left"
)

test_text = test.merge(
    test_transcripts,
    on="filename",
    how="left"
)

# Basic text statistics
train_text["word_count"] = (
    train_text["transcript"]
    .fillna("")
    .str.split()
    .str.len()
)

test_text["word_count"] = (
    test_text["transcript"]
    .fillna("")
    .str.split()
    .str.len()
)

train_text["char_count"] = (
    train_text["transcript"]
    .fillna("")
    .str.len()
)

test_text["char_count"] = (
    test_text["transcript"]
    .fillna("")
    .str.len()
)

print("=" * 70)
print("TRANSCRIPT QUALITY AUDIT")
print("=" * 70)

print("\nTrain transcript statistics:")
print(
    train_text["word_count"].describe()
)

print("\nTest transcript statistics:")
print(
    test_text["word_count"].describe()
)

print("\nEmpty transcripts:")
print(
    "Train:",
    (train_text["word_count"] == 0).sum()
)
print(
    "Test :",
    (test_text["word_count"] == 0).sum()
)

print("\nVery short transcripts (<10 words):")
print(
    "Train:",
    (train_text["word_count"] < 10).sum()
)
print(
    "Test :",
    (test_text["word_count"] < 10).sum()
)

print("\n" + "=" * 70)
print("SAMPLE TRANSCRIPTS")
print("=" * 70)

for _, row in train_text.sample(
    5,
    random_state=SEED
).iterrows():

    print(
        f"\nFilename: {row['filename']}"
        f" | Label: {row['label']}"
        f" | Words: {row['word_count']}"
    )

    print(row["transcript"][:500])

print("\n" + "=" * 70)
print("✅ TRANSCRIPT AUDIT COMPLETE")
print("=" * 70)

TRANSCRIPT QUALITY AUDIT

Train transcript statistics:
count    769.000000
mean      53.050715
std       33.609488
min        1.000000
25%       40.000000
50%       53.000000
75%       62.000000
max      444.000000
Name: word_count, dtype: float64

Test transcript statistics:
count    216.000000
mean      58.939815
std       25.371126
min        1.000000
25%       49.000000
50%       59.000000
75%       67.250000
max      247.000000
Name: word_count, dtype: float64

Empty transcripts:
Train: 0
Test : 0

Very short transcripts (<10 words):
Train: 47
Test : 5

SAMPLE TRANSCRIPTS

Filename: audio_5072.wav | Label: 0.0 | Words: 58
My goal is I want to earn sufficient money for me and my family and to clear all the loans. Actually, I really want to prepare a house to provide them with so much money. So to clear that loan, I want to earn money so my goal is to earn sufficient money for me and my family.

Filename: audio_708.wav | Label: 3.0 | Words: 40
My favorite place to visit is my local 

In [10]:
# ============================================================
# CELL 11 — CHAR TF-IDF + RIDGE OOF
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge

oof_char = np.zeros(len(train_text))
test_char = np.zeros(len(test_text))

for fold in range(N_FOLDS):

    train_idx = train_text["fold"].values != fold
    val_idx = train_text["fold"].values == fold

    X_train_text = train_text.loc[train_idx, "transcript"]
    X_val_text = train_text.loc[val_idx, "transcript"]

    y_train = train_text.loc[train_idx, "label"].values

    vectorizer = TfidfVectorizer(
        analyzer="char",
        ngram_range=(3, 5),
        min_df=2,
        max_features=50000,
        sublinear_tf=True
    )

    X_tr = vectorizer.fit_transform(X_train_text)
    X_val = vectorizer.transform(X_val_text)
    X_test = vectorizer.transform(
        test_text["transcript"]
    )

    model = Ridge(alpha=1.0)

    model.fit(X_tr, y_train)

    oof_char[val_idx] = model.predict(X_val)
    test_char += model.predict(X_test) / N_FOLDS

    fold_p, fold_rmse = evaluate_predictions(
        train_text.loc[val_idx, "label"].values,
        oof_char[val_idx]
    )

    print(
        f"Fold {fold}: "
        f"Pearson={fold_p:.4f} | "
        f"RMSE={fold_rmse:.4f}"
    )

print("\n" + "=" * 70)
print("CHAR TF-IDF + RIDGE — OVERALL OOF")
print("=" * 70)

char_pearson, char_rmse = evaluate_predictions(
    train_text["label"].values,
    oof_char
)

print(f"Pearson : {char_pearson:.4f}")
print(f"RMSE    : {char_rmse:.4f}")

print("\nOOF prediction range:")
print(
    f"{oof_char.min():.4f} → {oof_char.max():.4f}"
)

print("\n" + "=" * 70)
print("✅ CHAR TF-IDF BRANCH COMPLETE")
print("=" * 70)

Fold 0: Pearson=0.4610 | RMSE=1.2229
Fold 1: Pearson=0.5415 | RMSE=1.0560
Fold 2: Pearson=0.5172 | RMSE=1.0207
Fold 3: Pearson=0.5811 | RMSE=1.0384
Fold 4: Pearson=0.4245 | RMSE=1.0756

CHAR TF-IDF + RIDGE — OVERALL OOF
Pearson : 0.4950
RMSE    : 1.0852

OOF prediction range:
0.8707 → 4.6019

✅ CHAR TF-IDF BRANCH COMPLETE


In [11]:
# ============================================================
# CELL 12 — WORD TF-IDF + RIDGE OOF
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge

oof_word = np.zeros(len(train_text))
test_word = np.zeros(len(test_text))

for fold in range(N_FOLDS):

    train_idx = train_text["fold"].values != fold
    val_idx = train_text["fold"].values == fold

    X_train_text = train_text.loc[
        train_idx, "transcript"
    ]

    X_val_text = train_text.loc[
        val_idx, "transcript"
    ]

    y_train = train_text.loc[
        train_idx, "label"
    ].values

    vectorizer = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_features=50000,
        sublinear_tf=True
    )

    X_tr = vectorizer.fit_transform(X_train_text)
    X_val = vectorizer.transform(X_val_text)
    X_test = vectorizer.transform(
        test_text["transcript"]
    )

    model = Ridge(alpha=1.0)

    model.fit(X_tr, y_train)

    oof_word[val_idx] = model.predict(X_val)
    test_word += model.predict(X_test) / N_FOLDS

    fold_p, fold_rmse = evaluate_predictions(
        train_text.loc[val_idx, "label"].values,
        oof_word[val_idx]
    )

    print(
        f"Fold {fold}: "
        f"Pearson={fold_p:.4f} | "
        f"RMSE={fold_rmse:.4f}"
    )

print("\n" + "=" * 70)
print("WORD TF-IDF + RIDGE — OVERALL OOF")
print("=" * 70)

word_pearson, word_rmse = evaluate_predictions(
    train_text["label"].values,
    oof_word
)

print(f"Pearson : {word_pearson:.4f}")
print(f"RMSE    : {word_rmse:.4f}")

print("\nOOF prediction range:")
print(
    f"{oof_word.min():.4f} → {oof_word.max():.4f}"
)

print("\n" + "=" * 70)
print("✅ WORD TF-IDF BRANCH COMPLETE")
print("=" * 70)

Fold 0: Pearson=0.5027 | RMSE=1.1982
Fold 1: Pearson=0.5444 | RMSE=1.0589
Fold 2: Pearson=0.4646 | RMSE=1.0471
Fold 3: Pearson=0.5831 | RMSE=1.0297
Fold 4: Pearson=0.4420 | RMSE=1.0646

WORD TF-IDF + RIDGE — OVERALL OOF
Pearson : 0.4970
RMSE    : 1.0814

OOF prediction range:
0.8944 → 4.6550

✅ WORD TF-IDF BRANCH COMPLETE


In [12]:
# ============================================================
# CELL 13 — LINGUISTIC / GRAMMAR FEATURES + CATBOOST
# ============================================================

from collections import Counter
from catboost import CatBoostRegressor


def extract_linguistic_features(text):
    text = str(text).strip()

    words = re.findall(r"\b[\w']+\b", text.lower())
    sentences = [
        s.strip()
        for s in re.split(r"[.!?]+", text)
        if s.strip()
    ]

    word_count = len(words)
    char_count = len(text)

    unique_words = len(set(words))

    sentence_count = max(len(sentences), 1)

    avg_sentence_length = (
        word_count / sentence_count
    )

    avg_word_length = (
        np.mean([len(w) for w in words])
        if words else 0
    )

    word_lengths = [len(w) for w in words]

    word_length_std = (
        np.std(word_lengths)
        if word_lengths else 0
    )

    max_word_length = (
        max(word_lengths)
        if word_lengths else 0
    )

    long_word_ratio = (
        np.mean([len(w) >= 8 for w in words])
        if words else 0
    )

    short_word_ratio = (
        np.mean([len(w) <= 2 for w in words])
        if words else 0
    )

    type_token_ratio = (
        unique_words / word_count
        if word_count > 0 else 0
    )

    # Repeated words
    counts = Counter(words)

    repetition_ratio = (
        sum(c - 1 for c in counts.values() if c > 1)
        / word_count
        if word_count > 0 else 0
    )

    # Common spoken fillers
    fillers = {
        "uh", "um", "hmm", "like",
        "actually", "basically",
        "you know", "so"
    }

    filler_count = sum(
        words.count(f)
        for f in fillers
        if " " not in f
    )

    filler_ratio = (
        filler_count / word_count
        if word_count > 0 else 0
    )

    # Bigram / trigram repetition
    bigrams = list(zip(words[:-1], words[1:]))

    trigrams = list(
        zip(words[:-2], words[1:-1], words[2:])
    )

    bigram_repetition = (
        1 - len(set(bigrams)) / len(bigrams)
        if bigrams else 0
    )

    trigram_repetition = (
        1 - len(set(trigrams)) / len(trigrams)
        if trigrams else 0
    )

    return [
        word_count,
        char_count,
        sentence_count,
        avg_sentence_length,
        type_token_ratio,
        repetition_ratio,
        filler_ratio,
        long_word_ratio,
        short_word_ratio,
        avg_word_length,
        max_word_length,
        word_length_std,
        bigram_repetition,
        trigram_repetition,
    ]


ling_feature_names = [
    "word_count",
    "char_count",
    "sentence_count",
    "avg_sentence_length",
    "type_token_ratio",
    "repetition_ratio",
    "filler_ratio",
    "long_word_ratio",
    "short_word_ratio",
    "avg_word_length",
    "max_word_length",
    "word_length_std",
    "bigram_repetition",
    "trigram_repetition",
]


X_ling = np.array([
    extract_linguistic_features(t)
    for t in train_text["transcript"]
])

X_ling_test = np.array([
    extract_linguistic_features(t)
    for t in test_text["transcript"]
])

print("Feature matrix:", X_ling.shape)

oof_ling = np.zeros(len(train_text))
test_ling = np.zeros(len(test_text))

for fold in range(N_FOLDS):

    train_idx = train_text["fold"].values != fold
    val_idx = train_text["fold"].values == fold

    model = CatBoostRegressor(
        iterations=500,
        depth=5,
        learning_rate=0.03,
        loss_function="RMSE",
        verbose=False,
        random_seed=SEED
    )

    model.fit(
        X_ling[train_idx],
        train_text.loc[train_idx, "label"].values
    )

    oof_ling[val_idx] = model.predict(
        X_ling[val_idx]
    )

    test_ling += (
        model.predict(X_ling_test) / N_FOLDS
    )

    fold_p, fold_rmse = evaluate_predictions(
        train_text.loc[val_idx, "label"].values,
        oof_ling[val_idx]
    )

    print(
        f"Fold {fold}: "
        f"Pearson={fold_p:.4f} | "
        f"RMSE={fold_rmse:.4f}"
    )

print("\n" + "=" * 70)
print("LINGUISTIC FEATURES — OVERALL OOF")
print("=" * 70)

ling_pearson, ling_rmse = evaluate_predictions(
    train_text["label"].values,
    oof_ling
)

print(f"Pearson : {ling_pearson:.4f}")
print(f"RMSE    : {ling_rmse:.4f}")

print("\n" + "=" * 70)
print("✅ LINGUISTIC BRANCH COMPLETE")
print("=" * 70)

Feature matrix: (769, 14)
Fold 0: Pearson=0.3745 | RMSE=1.2632
Fold 1: Pearson=0.4763 | RMSE=1.0897
Fold 2: Pearson=0.3065 | RMSE=1.1467
Fold 3: Pearson=0.3825 | RMSE=1.1604
Fold 4: Pearson=0.3542 | RMSE=1.1264

LINGUISTIC FEATURES — OVERALL OOF
Pearson : 0.3712
RMSE    : 1.1588

✅ LINGUISTIC BRANCH COMPLETE


In [13]:
# ============================================================
# CELL 14 — TEXT OOF BLEND COMPARISON
# ============================================================

from itertools import product

predictions = {
    "char": oof_char,
    "word": oof_word,
    "ling": oof_ling
}

y_true = train_text["label"].values

results = []

# Char + Word only
for w_char in np.arange(0, 1.01, 0.05):

    w_word = 1.0 - w_char

    pred = (
        w_char * oof_char
        + w_word * oof_word
    )

    p, rmse = evaluate_predictions(
        y_true,
        pred
    )

    results.append({
        "weights": f"char={w_char:.2f}, word={w_word:.2f}",
        "pearson": p,
        "rmse": rmse
    })


# Char + Word + Linguistic
for w_char in np.arange(0, 1.01, 0.10):

    for w_word in np.arange(0, 1.01 - w_char, 0.10):

        w_ling = 1.0 - w_char - w_word

        pred = (
            w_char * oof_char
            + w_word * oof_word
            + w_ling * oof_ling
        )

        p, rmse = evaluate_predictions(
            y_true,
            pred
        )

        results.append({
            "weights": (
                f"char={w_char:.2f}, "
                f"word={w_word:.2f}, "
                f"ling={w_ling:.2f}"
            ),
            "pearson": p,
            "rmse": rmse
        })


results_df = pd.DataFrame(results)

print("=" * 70)
print("BEST TEXT BLENDS")
print("=" * 70)

print("\nTop by RMSE:")
print(
    results_df
    .sort_values("rmse")
    .head(10)
    .to_string(index=False)
)

print("\nTop by Pearson:")
print(
    results_df
    .sort_values("pearson", ascending=False)
    .head(10)
    .to_string(index=False)
)

BEST TEXT BLENDS

Top by RMSE:
                        weights  pearson     rmse
char=0.20, word=0.60, ling=0.20 0.528503 1.070228
char=0.30, word=0.50, ling=0.20 0.529401 1.070366
char=0.10, word=0.70, ling=0.20 0.526629 1.070478
char=0.10, word=0.60, ling=0.30 0.528265 1.070644
char=0.20, word=0.50, ling=0.30 0.529250 1.070758
char=0.40, word=0.40, ling=0.20 0.529267 1.070892
char=0.00, word=0.70, ling=0.30 0.526299 1.070918
char=0.00, word=0.80, ling=0.20 0.523843 1.071117
char=0.30, word=0.40, ling=0.30 0.529199 1.071261
char=0.50, word=0.30, ling=0.20 0.528066 1.071805

Top by Pearson:
                        weights  pearson     rmse
char=0.30, word=0.50, ling=0.20 0.529401 1.070366
char=0.40, word=0.40, ling=0.20 0.529267 1.070892
char=0.20, word=0.50, ling=0.30 0.529250 1.070758
char=0.30, word=0.40, ling=0.30 0.529199 1.071261
char=0.20, word=0.60, ling=0.20 0.528503 1.070228
char=0.10, word=0.60, ling=0.30 0.528265 1.070644
char=0.40, word=0.30, ling=0.30 0.528073 1.072150
ch

In [14]:
# ============================================================
# CELL 15 — FAST ACOUSTIC / PROSODY FEATURES + CATBOOST
# ============================================================

import librosa
import numpy as np
import time
from catboost import CatBoostRegressor


def extract_acoustic_features_fast(filepath):

    audio, sr = librosa.load(
        filepath,
        sr=16000,
        mono=True
    )

    duration = len(audio) / sr

    # --------------------------------------------------------
    # Energy
    # --------------------------------------------------------
    rms = librosa.feature.rms(
        y=audio,
        hop_length=512
    )[0]

    # --------------------------------------------------------
    # Zero Crossing
    # --------------------------------------------------------
    zcr = librosa.feature.zero_crossing_rate(
        audio,
        hop_length=512
    )[0]

    # --------------------------------------------------------
    # Spectral Features
    # --------------------------------------------------------
    centroid = librosa.feature.spectral_centroid(
        y=audio,
        sr=sr,
        hop_length=512
    )[0]

    bandwidth = librosa.feature.spectral_bandwidth(
        y=audio,
        sr=sr,
        hop_length=512
    )[0]

    rolloff = librosa.feature.spectral_rolloff(
        y=audio,
        sr=sr,
        hop_length=512
    )[0]

    # --------------------------------------------------------
    # MFCC
    # --------------------------------------------------------
    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sr,
        n_mfcc=13,
        hop_length=512
    )

    features = [
        duration,

        rms.mean(),
        rms.std(),
        rms.min(),
        rms.max(),

        zcr.mean(),
        zcr.std(),

        centroid.mean(),
        centroid.std(),

        bandwidth.mean(),
        bandwidth.std(),

        rolloff.mean(),
        rolloff.std()
    ]

    # MFCC mean + std
    for i in range(13):
        features.append(mfcc[i].mean())
        features.append(mfcc[i].std())

    # --------------------------------------------------------
    # FAST PITCH ESTIMATION
    # Using YIN instead of slow PYIN
    # --------------------------------------------------------
    try:

        f0 = librosa.yin(
            audio,
            fmin=70,
            fmax=400,
            sr=sr,
            frame_length=2048,
            hop_length=512
        )

        valid_f0 = f0[
            np.isfinite(f0) &
            (f0 >= 70) &
            (f0 <= 400)
        ]

        if len(valid_f0) > 0:

            pitch_mean = np.mean(valid_f0)
            pitch_std = np.std(valid_f0)
            pitch_min = np.min(valid_f0)
            pitch_max = np.max(valid_f0)

        else:

            pitch_mean = 0
            pitch_std = 0
            pitch_min = 0
            pitch_max = 0

        voiced_ratio = len(valid_f0) / len(f0)

    except Exception:

        pitch_mean = 0
        pitch_std = 0
        pitch_min = 0
        pitch_max = 0
        voiced_ratio = 0

    features.extend([
        pitch_mean,
        pitch_std,
        pitch_min,
        pitch_max,
        voiced_ratio
    ])

    # --------------------------------------------------------
    # Low-energy / pause proxy
    # --------------------------------------------------------
    energy_threshold = np.percentile(
        rms,
        20
    )

    low_energy_ratio = np.mean(
        rms <= energy_threshold
    )

    features.append(low_energy_ratio)

    return features


# ============================================================
# FEATURE EXTRACTION
# ============================================================

print("=" * 70)
print("FAST ACOUSTIC FEATURE EXTRACTION")
print("=" * 70)

start_time = time.time()

X_audio = np.array([
    extract_acoustic_features_fast(
        TRAIN_AUDIO_DIR / f
    )
    for f in train["filename"]
])

X_audio_test = np.array([
    extract_acoustic_features_fast(
        TEST_AUDIO_DIR / f
    )
    for f in test["filename"]
])

print("\nTrain feature matrix:", X_audio.shape)
print("Test feature matrix :", X_audio_test.shape)

print(
    "Extraction time:",
    round((time.time() - start_time) / 60, 2),
    "minutes"
)

print(
    "Missing values:",
    np.isnan(X_audio).sum()
)


# ============================================================
# OOF CATBOOST
# ============================================================

print("\n" + "=" * 70)
print("CATBOOST OOF")
print("=" * 70)

oof_audio = np.zeros(len(train))
test_audio = np.zeros(len(test))

for fold in range(N_FOLDS):

    train_idx = train["fold"].values != fold
    val_idx = train["fold"].values == fold

    model = CatBoostRegressor(
        iterations=500,
        depth=5,
        learning_rate=0.03,
        loss_function="RMSE",
        verbose=False,
        random_seed=SEED
    )

    model.fit(
        X_audio[train_idx],
        train.loc[train_idx, "label"].values
    )

    oof_audio[val_idx] = model.predict(
        X_audio[val_idx]
    )

    test_audio += (
        model.predict(X_audio_test) / N_FOLDS
    )

    fold_p, fold_rmse = evaluate_predictions(
        train.loc[val_idx, "label"].values,
        oof_audio[val_idx]
    )

    print(
        f"Fold {fold}: "
        f"Pearson={fold_p:.4f} | "
        f"RMSE={fold_rmse:.4f}"
    )


# ============================================================
# OVERALL
# ============================================================

print("\n" + "=" * 70)
print("FAST ACOUSTIC / PROSODY — OVERALL OOF")
print("=" * 70)

audio_pearson, audio_rmse = evaluate_predictions(
    train["label"].values,
    oof_audio
)

print(f"Pearson : {audio_pearson:.4f}")
print(f"RMSE    : {audio_rmse:.4f}")

print("\n" + "=" * 70)
print("✅ FAST ACOUSTIC BRANCH COMPLETE")
print("=" * 70)

FAST ACOUSTIC FEATURE EXTRACTION

Train feature matrix: (769, 45)
Test feature matrix : (216, 45)
Extraction time: 6.6 minutes
Missing values: 0

CATBOOST OOF
Fold 0: Pearson=0.8450 | RMSE=0.7268
Fold 1: Pearson=0.7921 | RMSE=0.7499
Fold 2: Pearson=0.7199 | RMSE=0.8172
Fold 3: Pearson=0.7798 | RMSE=0.7779
Fold 4: Pearson=0.6995 | RMSE=0.8506

FAST ACOUSTIC / PROSODY — OVERALL OOF
Pearson : 0.7734
RMSE    : 0.7857

✅ FAST ACOUSTIC BRANCH COMPLETE


In [15]:
# ============================================================
# CELL 16 — AUDIO + TEXT OOF BLENDING
# ============================================================

print("=" * 70)
print("AUDIO + TEXT OOF BLENDING")
print("=" * 70)

best_score = None
best_weights = None

# ------------------------------------------------------------
# Grid search
# Audio + Word + Char + Linguistic
# ------------------------------------------------------------

for w_audio in np.arange(0.50, 1.01, 0.05):

    remaining = 1.0 - w_audio

    for w_word in np.arange(0.00, remaining + 0.001, 0.05):

        for w_char in np.arange(0.00, remaining - w_word + 0.001, 0.05):

            w_ling = remaining - w_word - w_char

            if w_ling < -1e-9:
                continue

            pred = (
                w_audio * oof_audio
                + w_word * oof_word
                + w_char * oof_char
                + w_ling * oof_ling
            )

            p, rmse = evaluate_predictions(
                y,
                pred
            )

            # Primary objective = RMSE
            if best_score is None or rmse < best_score:
                best_score = rmse
                best_weights = (
                    w_audio,
                    w_word,
                    w_char,
                    w_ling,
                    p,
                    rmse
                )

print("\nBest RMSE blend:")
print(
    f"Audio={best_weights[0]:.2f}, "
    f"Word={best_weights[1]:.2f}, "
    f"Char={best_weights[2]:.2f}, "
    f"Ling={best_weights[3]:.2f}"
)

print(f"Pearson : {best_weights[4]:.4f}")
print(f"RMSE    : {best_weights[5]:.4f}")


# ------------------------------------------------------------
# Compare important baselines
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("IMPORTANT COMPARISONS")
print("=" * 70)

print_scores(
    "Audio only",
    y,
    oof_audio
)

text_pred = (
    0.30 * oof_char
    + 0.50 * oof_word
    + 0.20 * oof_ling
)

print_scores(
    "Text blend",
    y,
    text_pred
)

best_pred = (
    best_weights[0] * oof_audio
    + best_weights[1] * oof_word
    + best_weights[2] * oof_char
    + best_weights[3] * oof_ling
)

print_scores(
    "Best Audio + Text",
    y,
    best_pred
)

print("\n✅ BLENDING COMPLETE")

AUDIO + TEXT OOF BLENDING

Best RMSE blend:
Audio=0.85, Word=0.10, Char=0.00, Ling=0.05
Pearson : 0.7888
RMSE    : 0.7777

IMPORTANT COMPARISONS
Audio only
Pearson : 0.7734
RMSE    : 0.7857
Text blend
Pearson : 0.5294
RMSE    : 1.0704
Best Audio + Text
Pearson : 0.7888
RMSE    : 0.7777

✅ BLENDING COMPLETE


In [16]:
# ============================================================
# CELL 17 — WHISPER ENCODER EMBEDDINGS + RIDGE
# FIXED VERSION
# ============================================================

import os
import time
import numpy as np
import torch
import librosa

from transformers import WhisperProcessor, WhisperModel
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline


# ============================================================
# SETTINGS
# ============================================================

WHISPER_TRAIN_CACHE = CACHE_DIR / "whisper_encoder_train.npy"
WHISPER_TEST_CACHE  = CACHE_DIR / "whisper_encoder_test.npy"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 70)
print("WHISPER ENCODER EMBEDDINGS")
print("=" * 70)

print("Device:", DEVICE)


# ============================================================
# LOAD WHISPER ENCODER WITH NEW VARIABLE NAMES
# ============================================================

print("\nLoading Whisper encoder...")

whisper_processor = WhisperProcessor.from_pretrained(
    "openai/whisper-small"
)

whisper_encoder = WhisperModel.from_pretrained(
    "openai/whisper-small"
).to(DEVICE)

whisper_encoder.eval()

print("✅ Whisper encoder loaded")


# ============================================================
# EMBEDDING FUNCTION
# ============================================================

def extract_whisper_embeddings(
    filepaths,
    cache_path,
    batch_size=4
):

    # --------------------------------------------------------
    # Load cache if available
    # --------------------------------------------------------

    if cache_path.exists():

        print("\nLoading cached embeddings:")
        print(cache_path)

        embeddings = np.load(cache_path)

        print(
            "Shape:",
            embeddings.shape
        )

        return embeddings

    all_embeddings = []

    total = len(filepaths)

    start_time = time.time()

    # --------------------------------------------------------
    # Process audio
    # --------------------------------------------------------

    for start in range(
        0,
        total,
        batch_size
    ):

        batch_files = filepaths[
            start:start + batch_size
        ]

        batch_audio = []

        for filepath in batch_files:

            audio, sr = librosa.load(
                filepath,
                sr=16000,
                mono=True
            )

            batch_audio.append(audio)

        # ----------------------------------------------------
        # Whisper processor
        # ----------------------------------------------------

        inputs = whisper_processor(
            batch_audio,
            sampling_rate=16000,
            return_tensors="pt",
            padding=True
        )

        input_features = inputs.input_features.to(
            DEVICE
        )

        # ----------------------------------------------------
        # Encoder
        # ----------------------------------------------------

        with torch.no_grad():

            encoder_output = whisper_encoder.encoder(
                input_features
            )

            hidden = encoder_output.last_hidden_state

            # Mean pool over time
            pooled = hidden.mean(
                dim=1
            )

        all_embeddings.append(
            pooled.cpu().numpy()
        )

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        done = min(
            start + batch_size,
            total
        )

        elapsed = time.time() - start_time

        print(
            f"\rProcessed {done}/{total} "
            f"({done/total*100:.1f}%) | "
            f"Elapsed: {elapsed/60:.1f} min",
            end=""
        )

    print()

    embeddings = np.vstack(
        all_embeddings
    ).astype(np.float32)

    # Save cache
    np.save(
        cache_path,
        embeddings
    )

    print(
        "Saved:",
        cache_path
    )

    print(
        "Shape:",
        embeddings.shape
    )

    return embeddings


# ============================================================
# FILE PATHS
# ============================================================

train_audio_paths = [
    TRAIN_AUDIO_DIR / f
    for f in train["filename"]
]

test_audio_paths = [
    TEST_AUDIO_DIR / f
    for f in test["filename"]
]


# ============================================================
# TRAIN EMBEDDINGS
# ============================================================

print("\n" + "=" * 70)
print("TRAIN WHISPER EMBEDDINGS")
print("=" * 70)

start_time = time.time()

X_whisper = extract_whisper_embeddings(
    train_audio_paths,
    WHISPER_TRAIN_CACHE,
    batch_size=4
)

print(
    "\nTrain extraction time:",
    round(
        (time.time() - start_time) / 60,
        2
    ),
    "minutes"
)


# ============================================================
# TEST EMBEDDINGS
# ============================================================

print("\n" + "=" * 70)
print("TEST WHISPER EMBEDDINGS")
print("=" * 70)

start_time = time.time()

X_whisper_test = extract_whisper_embeddings(
    test_audio_paths,
    WHISPER_TEST_CACHE,
    batch_size=4
)

print(
    "\nTest extraction time:",
    round(
        (time.time() - start_time) / 60,
        2
    ),
    "minutes"
)


# ============================================================
# CHECK
# ============================================================

print("\n" + "=" * 70)
print("EMBEDDING MATRICES")
print("=" * 70)

print(
    "Train:",
    X_whisper.shape
)

print(
    "Test :",
    X_whisper_test.shape
)

print(
    "NaNs :",
    np.isnan(X_whisper).sum()
)


# ============================================================
# 5-FOLD RIDGE OOF
# ============================================================

print("\n" + "=" * 70)
print("WHISPER RIDGE — 5-FOLD OOF")
print("=" * 70)

oof_whisper = np.zeros(
    len(train)
)

test_whisper = np.zeros(
    len(test)
)

for fold in range(N_FOLDS):

    train_idx = (
        train["fold"].values != fold
    )

    val_idx = (
        train["fold"].values == fold
    )

    ridge_model = make_pipeline(
        StandardScaler(),
        Ridge(alpha=10.0)
    )

    ridge_model.fit(
        X_whisper[train_idx],
        train.loc[
            train_idx,
            "label"
        ].values
    )

    oof_whisper[val_idx] = (
        ridge_model.predict(
            X_whisper[val_idx]
        )
    )

    test_whisper += (
        ridge_model.predict(
            X_whisper_test
        ) / N_FOLDS
    )

    fold_p, fold_rmse = evaluate_predictions(
        train.loc[
            val_idx,
            "label"
        ].values,
        oof_whisper[val_idx]
    )

    print(
        f"Fold {fold}: "
        f"Pearson={fold_p:.4f} | "
        f"RMSE={fold_rmse:.4f}"
    )


# ============================================================
# OVERALL
# ============================================================

print("\n" + "=" * 70)
print("WHISPER ENCODER — OVERALL OOF")
print("=" * 70)

whisper_pearson, whisper_rmse = (
    evaluate_predictions(
        y,
        oof_whisper
    )
)

print(
    f"Pearson : {whisper_pearson:.4f}"
)

print(
    f"RMSE    : {whisper_rmse:.4f}"
)

print("\n" + "=" * 70)
print("✅ WHISPER ENCODER BRANCH COMPLETE")
print("=" * 70)

WHISPER ENCODER EMBEDDINGS
Device: cuda

Loading Whisper encoder...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

✅ Whisper encoder loaded

TRAIN WHISPER EMBEDDINGS
Processed 769/769 (100.0%) | Elapsed: 1.7 min
Saved: /kaggle/working/cache/whisper_encoder_train.npy
Shape: (769, 768)

Train extraction time: 1.73 minutes

TEST WHISPER EMBEDDINGS
Processed 216/216 (100.0%) | Elapsed: 0.5 min
Saved: /kaggle/working/cache/whisper_encoder_test.npy
Shape: (216, 768)

Test extraction time: 0.48 minutes

EMBEDDING MATRICES
Train: (769, 768)
Test : (216, 768)
NaNs : 0

WHISPER RIDGE — 5-FOLD OOF
Fold 0: Pearson=0.8275 | RMSE=0.8022
Fold 1: Pearson=0.7659 | RMSE=0.8414
Fold 2: Pearson=0.7582 | RMSE=0.7834
Fold 3: Pearson=0.8278 | RMSE=0.7626
Fold 4: Pearson=0.7722 | RMSE=0.8115

WHISPER ENCODER — OVERALL OOF
Pearson : 0.7922
RMSE    : 0.8007

✅ WHISPER ENCODER BRANCH COMPLETE


In [17]:
# ============================================================
# CELL 18 — AUDIO + WHISPER + TEXT BLENDING
# ============================================================

print("=" * 70)
print("AUDIO + WHISPER + TEXT OOF BLENDING")
print("=" * 70)

best_rmse = float("inf")
best_result = None

# ------------------------------------------------------------
# Grid search
# Audio + Whisper + Word + Linguistic
# ------------------------------------------------------------

for w_audio in np.arange(0.40, 1.01, 0.05):

    for w_whisper in np.arange(
        0.00,
        1.01 - w_audio,
        0.05
    ):

        remaining = 1.0 - w_audio - w_whisper

        if remaining < -1e-9:
            continue

        # Split remaining between Word and Linguistic
        for w_word in np.arange(
            0.00,
            remaining + 0.001,
            0.05
        ):

            w_ling = remaining - w_word

            if w_ling < -1e-9:
                continue

            pred = (
                w_audio * oof_audio
                + w_whisper * oof_whisper
                + w_word * oof_word
                + w_ling * oof_ling
            )

            p, rmse = evaluate_predictions(
                y,
                pred
            )

            if rmse < best_rmse:

                best_rmse = rmse

                best_result = (
                    w_audio,
                    w_whisper,
                    w_word,
                    w_ling,
                    p,
                    rmse
                )


# ------------------------------------------------------------
# Best result
# ------------------------------------------------------------

print("\nBest RMSE blend:")

print(
    f"Audio   = {best_result[0]:.2f}"
)

print(
    f"Whisper = {best_result[1]:.2f}"
)

print(
    f"Word    = {best_result[2]:.2f}"
)

print(
    f"Ling    = {best_result[3]:.2f}"
)

print(
    f"Pearson : {best_result[4]:.4f}"
)

print(
    f"RMSE    : {best_result[5]:.4f}"
)


# ------------------------------------------------------------
# Construct best prediction
# ------------------------------------------------------------

best_pred_awt = (
    best_result[0] * oof_audio
    + best_result[1] * oof_whisper
    + best_result[2] * oof_word
    + best_result[3] * oof_ling
)


# ------------------------------------------------------------
# Compare branches
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("COMPARISON")
print("=" * 70)

print_scores(
    "Audio only",
    y,
    oof_audio
)

print_scores(
    "Whisper only",
    y,
    oof_whisper
)

print_scores(
    "Audio + Text",
    y,
    0.85 * oof_audio
    + 0.10 * oof_word
    + 0.05 * oof_ling
)

print_scores(
    "Audio + Whisper + Text",
    y,
    best_pred_awt
)

print("\n" + "=" * 70)
print("✅ THREE-BRANCH BLEND COMPLETE")
print("=" * 70)

AUDIO + WHISPER + TEXT OOF BLENDING

Best RMSE blend:
Audio   = 0.50
Whisper = 0.45
Word    = 0.05
Ling    = -0.00
Pearson : 0.8478
RMSE    : 0.6627

COMPARISON
Audio only
Pearson : 0.7734
RMSE    : 0.7857
Whisper only
Pearson : 0.7922
RMSE    : 0.8007
Audio + Text
Pearson : 0.7888
RMSE    : 0.7777
Audio + Whisper + Text
Pearson : 0.8478
RMSE    : 0.6627

✅ THREE-BRANCH BLEND COMPLETE


In [18]:
# ============================================================
# CELL 19 — CURRENT BEST TEST PREDICTION
# ============================================================

FINAL_W_AUDIO = 0.50
FINAL_W_WHISPER = 0.45
FINAL_W_WORD = 0.05

final_test_pred = (
    FINAL_W_AUDIO * test_audio
    + FINAL_W_WHISPER * test_whisper
    + FINAL_W_WORD * test_word
)

# Keep predictions within valid 0–5 range
final_test_pred = np.clip(
    final_test_pred,
    0,
    5
)

print("=" * 70)
print("CURRENT BEST TEST PREDICTIONS")
print("=" * 70)

print("Prediction count:", len(final_test_pred))
print("Min:", final_test_pred.min())
print("Max:", final_test_pred.max())
print("Mean:", final_test_pred.mean())
print("Std :", final_test_pred.std())

print("\nFirst 10 predictions:")
print(final_test_pred[:10])

print("\n✅ TEST PREDICTIONS READY")

CURRENT BEST TEST PREDICTIONS
Prediction count: 216
Min: 1.5041195185975147
Max: 4.862240952411254
Mean: 3.28800480391027
Std : 0.6162069687334633

First 10 predictions:
[3.18986208 4.32161633 4.28455081 2.2939327  2.80516603 3.30839443
 2.47195075 2.30741492 3.54960289 3.57227559]

✅ TEST PREDICTIONS READY


In [19]:
# ============================================================
# CELL 20 — CREATE + VALIDATE SUBMISSION
# ============================================================

submission = pd.DataFrame({
    "filename": test["filename"].values,
    "label": final_test_pred
})

submission_path = WORK_DIR / "submission_v2_audio_whisper_word.csv"

submission.to_csv(
    submission_path,
    index=False
)

print("=" * 70)
print("SUBMISSION CREATED")
print("=" * 70)

print("Path:", submission_path)
print("Shape:", submission.shape)

print("\nColumns:")
print(submission.columns.tolist())

print("\nFirst 10 rows:")
print(submission.head(10))

print("\nValidation:")
print(
    "Correct row count:",
    len(submission) == len(test)
)

print(
    "Filename order correct:",
    np.array_equal(
        submission["filename"].values,
        test["filename"].values
    )
)

print(
    "Missing predictions:",
    submission["label"].isna().sum()
)

print(
    "Predictions outside 0–5:",
    (
        (submission["label"] < 0) |
        (submission["label"] > 5)
    ).sum()
)

print("\nSaved to:")
print(submission_path)

print("\n✅ SUBMISSION READY")

SUBMISSION CREATED
Path: /kaggle/working/submission_v2_audio_whisper_word.csv
Shape: (216, 2)

Columns:
['filename', 'label']

First 10 rows:
        filename     label
0  audio_128.wav  3.189862
1  audio_156.wav  4.321616
2   audio_19.wav  4.284551
3  audio_108.wav  2.293933
4  audio_206.wav  2.805166
5  audio_161.wav  3.308394
6  audio_215.wav  2.471951
7  audio_213.wav  2.307415
8   audio_11.wav  3.549603
9  audio_155.wav  3.572276

Validation:
Correct row count: True
Filename order correct: True
Missing predictions: 0
Predictions outside 0–5: 0

Saved to:
/kaggle/working/submission_v2_audio_whisper_word.csv

✅ SUBMISSION READY


In [20]:
# Create the exact filename required by Kaggle
submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print("✅ Created:")
print("/kaggle/working/submission.csv")

✅ Created:
/kaggle/working/submission.csv


In [21]:
import pandas as pd
from pathlib import Path

old_file = Path("/kaggle/working/submission_v2_audio_whisper_word.csv")
new_file = Path("/kaggle/working/submission.csv")

submission = pd.read_csv(old_file)

submission.to_csv(new_file, index=False)

print("✅ Created:", new_file)
print("Shape:", submission.shape)
print(submission.head())

✅ Created: /kaggle/working/submission.csv
Shape: (216, 2)
        filename     label
0  audio_128.wav  3.189862
1  audio_156.wav  4.321616
2   audio_19.wav  4.284551
3  audio_108.wav  2.293933
4  audio_206.wav  2.805166
